# Section 0 — Configuration
Pipeline settings for v3 (Large-Scale Hybrid AST-GNN). Includes multiprocessing options, chunked disk caching, and CodeBERT LM integration.

In [3]:
# ========================== SECTION 0: CONFIGURATION ==========================
DEBUG = True               # True → sample 1000 rows; False → full dataset
BATCH_SIZE = 32
HIDDEN_DIM = 128
EMBEDDING_DIM = 32         # Learned node-type embedding dim
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3
DROPOUT = 0.3
MAX_NODES = 512
CACHE_DIR = "./graph_cache_v3"
PATIENCE = 5
WEIGHT_DECAY = 1e-4
DEBUG_SAMPLE_SIZE = 1000
RANDOM_SEED = 42
TRUNCATION_STRATEGY = "dfs"

# Large-scale & LM adjustments
NUM_WORKERS = 4             # Multiprocessing workers for graph generation
CHUNK_SIZE = 5000           # Number of graphs per .pt file
USE_TOKEN_FEATURES = True   # Surface-level heuristics
USE_CODEBERT = True         # Extract CodeBERT pooler embeddings
CODEBERT_MODEL = "microsoft/codebert-base"

NUM_CLASSES = 2             # Task A: binary

import os, pathlib, hashlib
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Config: DEBUG={DEBUG}, GNN_HIDDEN={HIDDEN_DIM}, CACHE_DIR={CACHE_DIR}")
print(f"Workers={NUM_WORKERS}, ChunkSize={CHUNK_SIZE}, CodeBERT={USE_CODEBERT}")

Config: DEBUG=True, GNN_HIDDEN=128, CACHE_DIR=./graph_cache_v3
Workers=4, ChunkSize=5000, CodeBERT=True


# Section 1 — Install Dependencies
Adds `transformers` for CodeBERT and `joblib` for multiprocessing graph generation.

In [4]:
# ========================== SECTION 1: INSTALL DEPENDENCIES ==========================
import subprocess, sys, importlib.util

def install_if_missing(package, pip_name=None):
    if importlib.util.find_spec(package) is None:
        name = pip_name or package
        print(f"Installing {name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", name])
    else:
        print(f"  ✓ {package}")

for pkg, pip in [("numpy",None),("pandas",None),("sklearn","scikit-learn"),
                 ("tqdm",None),("matplotlib",None),("datasets",None),
                 ("joblib",None),("transformers",None)]:
    install_if_missing(pkg, pip)

install_if_missing("torch")
install_if_missing("tree_sitter", "tree-sitter>=0.21.0,<1.0")

for imp, pip in [("tree_sitter_python","tree-sitter-python"),
                 ("tree_sitter_cpp","tree-sitter-cpp"),
                 ("tree_sitter_java","tree-sitter-java"),
                 ("tree_sitter_javascript","tree-sitter-javascript"),
                 ("tree_sitter_go","tree-sitter-go"),
                 ("tree_sitter_c_sharp","tree-sitter-c-sharp"),
                 ("tree_sitter_c","tree-sitter-c")]:
    install_if_missing(imp, pip)

if importlib.util.find_spec("torch_geometric") is None:
    import torch
    print("Installing torch-geometric...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])
    except subprocess.CalledProcessError:
        tv = torch.__version__.split("+")[0]
        tag = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() and torch.version.cuda else "cpu"
        url = f"https://data.pyg.org/whl/torch-{tv}+{tag}.html"
        print(f"  Retrying with --find-links {url}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric", "--find-links", url])
else:
    print("  ✓ torch_geometric")

import torch, torch_geometric
print(f"\n✓ Dependencies installed. torch={torch.__version__}, pyg={torch_geometric.__version__}")

  ✓ numpy
  ✓ pandas
  ✓ sklearn
  ✓ tqdm
  ✓ matplotlib
  ✓ datasets
  ✓ joblib
  ✓ transformers
  ✓ torch
  ✓ tree_sitter
  ✓ tree_sitter_python
  ✓ tree_sitter_cpp
  ✓ tree_sitter_java
  ✓ tree_sitter_javascript
  ✓ tree_sitter_go
  ✓ tree_sitter_c_sharp
  ✓ tree_sitter_c
  ✓ torch_geometric

✓ Dependencies installed. torch=2.10.0, pyg=2.7.0


# Section 2 — Imports, Setup, & CodeBERT
Includes reproducibility and auto-device detection.  
Initializes the Language Model pipeline for feature extraction.

In [5]:
# ========================== SECTION 2: IMPORTS & SETUP ==========================
import os, sys, json, random, time, warnings, hashlib, re, platform
from collections import Counter, defaultdict, deque
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from joblib import Parallel, delayed

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

import torch_geometric
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool, GlobalAttention

from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

from tree_sitter import Language, Parser
import tree_sitter_python, tree_sitter_cpp, tree_sitter_java
import tree_sitter_javascript, tree_sitter_go, tree_sitter_c_sharp, tree_sitter_c

from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings("ignore")

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(RANDOM_SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device : {device}  |  OS: {platform.system()} {platform.machine()}")

# ---- Initialize CodeBERT (Used heavily in Graph construction) ----
if USE_CODEBERT:
    print(f"Loading {CODEBERT_MODEL} for hybrid feature extraction...")
    lm_tokenizer = AutoTokenizer.from_pretrained(CODEBERT_MODEL)
    lm_model = AutoModel.from_pretrained(CODEBERT_MODEL).to(device)
    lm_model.eval()
    CODEBERT_DIM = lm_model.config.hidden_size
    print(f"✓ Language Model ready (dim={CODEBERT_DIM})")
else:
    lm_tokenizer = None
    lm_model = None
    CODEBERT_DIM = 0

Device : mps  |  OS: Darwin arm64
Loading microsoft/codebert-base for hybrid feature extraction...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6300.65it/s]


✓ Language Model ready (dim=768)


# Section 3 — Dataset Loading & Leakage Detection
Loads data, preserves original `id`, computes majority class, and explicitly checks for cross-split duplicate snippets.

In [ ]:
# ========================== SECTION 3: DATASET LOADING & DOWNLOADING ==========================

import os, pathlib, hashlib
import pandas as pd
from sklearn.model_selection import train_test_split

def load_or_fetch_task_a():
    """Attempts to load data locally or from Kaggle mounts; if missing, downloads via API."""
    local_dir = "./task_a"
    local_train = os.path.join(local_dir, "train.parquet")
    local_val = os.path.join(local_dir, "validation.parquet")
    
    # 1. Check if we already physically have it locally
    if os.path.exists(local_train):
        print(f"Loading from local cache: {local_train}")
        return pd.read_parquet(local_train), pd.read_parquet(local_val)

    # 2. Check Kaggle paths explicitly from Task-A starter kit (if running on Kaggle server)
    kaggle_explicit_train = "/kaggle/input/sem-eval-2026-task-13-subtask-a/Task_A/train.parquet"
    kaggle_explicit_val = "/kaggle/input/sem-eval-2026-task-13-subtask-a/Task_A/validation.parquet"
    if os.path.exists(kaggle_explicit_train):
        print(f"Loading direct from Kaggle cluster: {kaggle_explicit_train}")
        val_df = pd.read_parquet(kaggle_explicit_val) if os.path.exists(kaggle_explicit_val) else None
        return pd.read_parquet(kaggle_explicit_train), val_df

    # 3. IF MISSING: Actively fetch via HuggingFace API and SAVE to local disk
    print("Files not found locally. Fetching dataset via HuggingFace API...")
    try:
        from datasets import load_dataset
        # FIXED: Subset string is "A", not "task_a"
        ds = load_dataset("DaniilOr/SemEval-2026-Task13", "A")
        
        train_df = ds["train"].to_pandas()
        val_df = ds["validation"].to_pandas() if "validation" in ds else None
        
        # Save them locally so we only download them once!
        os.makedirs(local_dir, exist_ok=True)
        print(f"Saving downloaded data to {local_dir}/ ...")
        train_df.to_parquet(local_train, index=False)
        if val_df is not None:
            val_df.to_parquet(local_val, index=False)
            
        return train_df, val_df
        
    except ImportError:
        raise ImportError("Please run `pip install datasets` to download HuggingFace files.")
    except Exception as e:
        raise RuntimeError(f"HuggingFace API Fetch failed: {e}\nPlease check your internet connection.")

def check_leakage(train_df, val_df, n_chars=200):
    if val_df is None: return
    t_h = set(hashlib.md5(s[:n_chars].encode()).hexdigest() for s in train_df["code"].dropna())
    v_h = set(hashlib.md5(s[:n_chars].encode()).hexdigest() for s in val_df["code"].dropna())
    overlap = t_h & v_h
    pct = 100.0 * len(overlap) / max(len(v_h), 1)
    if pct > 1.0: print(f"⚠ LEAKAGE: {len(overlap)} val samples ({pct:.1f}%) near-duplicate in train.")
    else: print(f"✓ Leakage check passed: {pct:.2f}% overlap")

# Execute fetch & load
train_df, val_df = load_or_fetch_task_a()

# Preserve IDs
if "id" not in train_df.columns: train_df["id"] = range(len(train_df))
if val_df is not None and "id" not in val_df.columns: val_df["id"] = range(len(val_df))

# Clean Null/Empty Snippets
for name, df in [("train", train_df), ("val", val_df)]:
    if df is None: continue
    mask = df["code"].notna() & (df["code"].str.strip() != "")
    if name == "train": train_df = train_df[mask].reset_index(drop=True)
    else: val_df = val_df[mask].reset_index(drop=True)

check_leakage(train_df, val_df)

if DEBUG:
    train_df, _ = train_test_split(train_df, train_size=min(DEBUG_SAMPLE_SIZE, len(train_df)), stratify=train_df["label"], random_state=RANDOM_SEED)
    train_df = train_df.reset_index(drop=True)
    if val_df is not None and len(val_df) > DEBUG_SAMPLE_SIZE // 2:
        val_df, _ = train_test_split(val_df, train_size=DEBUG_SAMPLE_SIZE // 2, stratify=val_df["label"], random_state=RANDOM_SEED)
        val_df = val_df.reset_index(drop=True)

MAJORITY_CLASS = int(train_df["label"].mode().iloc[0])
print(f"\nTrain: {len(train_df):,} | Val: {len(val_df) if val_df is not None else 0:,} | Majority: {MAJORITY_CLASS}")


Files not found locally. Fetching dataset via HuggingFace API...
Saving downloaded data to ./task_a/ ...
⚠ LEAKAGE: 1909 val samples (1.9%) near-duplicate in train.

Train: 1,000 | Val: 500 | Majority: 1


# Section 4 — AST Parser Setup
Version-safe wrapper + Generalized Semantic mapping. We intentionally map `identifier`, `integer`, etc to generalized tokens so that ASTs from Go/PHP align closely with Python/Java structures.

In [12]:
# ========================== SECTION 4: AST PARSER SETUP ==========================

def _make_parser(lang_module):
    lang_obj = Language(lang_module.language())
    try: return Parser(lang_obj)
    except TypeError: pass
    try: p = Parser(); p.language = lang_obj; return p
    except Exception: pass
    try: p = Parser(); p.set_language(lang_obj); return p
    except Exception as e: raise RuntimeError(e)

def build_parsers():
    modules = {"python": tree_sitter_python, "cpp": tree_sitter_cpp, "java": tree_sitter_java,
               "javascript": tree_sitter_javascript, "go": tree_sitter_go,
               "c_sharp": tree_sitter_c_sharp, "c": tree_sitter_c}
    parsers = {}
    for name, mod in modules.items():
        try: parsers[name] = _make_parser(mod)
        except Exception as e: print(f"  ⚠ {name}: {e}")
    print(f"✓ {len(parsers)} parsers ready.")
    return parsers

parsers = build_parsers()
_LANG_MAP = {"python":"python","py":"python","c++":"cpp","cpp":"cpp","java":"java","javascript":"javascript","js":"javascript",
             "go":"go","golang":"go","c#":"c_sharp","csharp":"c_sharp","c_sharp":"c_sharp","c":"c"}

def parse_code(code_str, language, parsers_dict):
    try:
        key = _LANG_MAP.get((language or "").strip().lower())
        if key not in parsers_dict: return None
        return parsers_dict[key].parse(code_str if isinstance(code_str, bytes) else code_str.encode("utf-8")).root_node
    except: return None

# Cross-language vocabulary smoothing
def normalize_node_type(t: str) -> str:
    """Reduce AST type fragmentation by combining exact semantic matches across languages."""
    t = t.lower()
    if "identifier" in t or "name" in t: return "IDENTIFIER"
    if "string" in t: return "STRING_LITERAL"
    if "number" in t or "integer" in t or "float" in t: return "NUMERIC_LITERAL"
    if "function" in t or "method" in t: return "FUNCTION_DEF"
    if "comment" in t: return "COMMENT"
    return t

# Smoke check
root = parse_code("def foo(): pass", "python", parsers)
assert root and root.child_count > 0, "Parser failed"
print("✓ Parsers functional.")

✓ 7 parsers ready.
✓ Parsers functional.


# Section 5 — AST to Graph Conversion & Feature Extraction
Upgraded for v3:
- Applies Semantic Cross-Language node norm
- Token-level heuristic features
- Generates base graph components (edges + normalized node counts)

In [13]:
# ========================== SECTION 5: AST TO GRAPH CONVERSION ==========================

def build_node_vocab(df, parsers_dict, max_samples=None):
    node_types = set()
    n = min(max_samples or len(df), len(df))
    for idx in tqdm(range(n), desc="Building Vocab"):
        row = df.iloc[idx]
        root = parse_code(row["code"], row.get("language", "python"), parsers_dict)
        if not root: continue
        stack = [root]
        while stack:
            nd = stack.pop()
            node_types.add(normalize_node_type(nd.type))
            stack.extend(nd.children)
    vocab = {"<UNK>": 0}
    for i, t in enumerate(sorted(node_types), 1): vocab[t] = i
    print(f"✓ Node vocab: {len(vocab)} types.")
    return vocab

def _subtree_sizes(node, cache=None):
    if cache is None: cache = {}
    nid = id(node)
    if nid in cache: return cache[nid]
    s = 1 + sum(_subtree_sizes(c, cache) for c in node.children)
    cache[nid] = s
    return s

def ast_to_graph(root_node, node_vocab, max_nodes=512, strategy="dfs"):
    # Collect nodes
    collected = []
    size_cache = {}
    _subtree_sizes(root_node, size_cache)
    
    if strategy == "dfs":
        stack = deque([(root_node, 0, -1, 0)])
        while stack and len(collected) < max_nodes:
            node, depth, pidx, pos = stack.pop()
            cur = len(collected)
            collected.append((node, depth, pidx, pos))
            children = list(node.children)
            for ci in range(len(children) - 1, -1, -1):
                if len(collected) + len(stack) < max_nodes:
                    stack.append((children[ci], depth + 1, cur, ci))
    else:  # BFS
        queue = deque([(root_node, 0, -1, 0)])
        while queue and len(collected) < max_nodes:
            node, depth, pidx, pos = queue.popleft()
            cur = len(collected)
            collected.append((node, depth, pidx, pos))
            for ci, child in enumerate(node.children):
                if len(collected) + len(queue) < max_nodes:
                    queue.append((child, depth + 1, cur, ci))

    if not collected: return None

    n = len(collected)
    src, dst = [], []
    parent_children = defaultdict(list)

    for cur, (_, _, pidx, _) in enumerate(collected):
        if pidx >= 0:
            src.extend([pidx, cur]); dst.extend([cur, pidx])
        parent_children[pidx].append(cur)

    # Sibling edges
    for pidx, children_ids in parent_children.items():
        if pidx < 0: continue
        for i in range(len(children_ids) - 1):
            a, b = children_ids[i], children_ids[i + 1]
            src.extend([a, b]); dst.extend([b, a])

    # Sequential edges
    for i in range(n - 1):
        src.extend([i, i + 1]); dst.extend([i + 1, i])

    max_depth = max(d for _, d, _, _ in collected) or 1
    max_children = max(nd.child_count for nd, _, _, _ in collected) or 1
    total_nodes = size_cache.get(id(root_node), n)

    type_ids, cont_fe = [], []
    for node, depth, pidx, pos in collected:
        type_ids.append(node_vocab.get(normalize_node_type(node.type), 0))
        parent_nchildren = collected[pidx][0].child_count if pidx >= 0 else 1
        cont_fe.append([
            depth / max_depth,
            node.child_count / max_children,
            1.0 if node.child_count == 0 else 0.0,
            1.0 if node.is_named else 0.0,
            size_cache.get(id(node), 1) / total_nodes,
            pos / max(parent_nchildren - 1, 1),
        ])

    return {
        "type_ids": torch.tensor(type_ids, dtype=torch.long),
        "cont_features": torch.tensor(cont_fe, dtype=torch.float),
        "edge_index": torch.tensor([src, dst], dtype=torch.long) if src else torch.tensor([[0],[0]], dtype=torch.long),
        "num_nodes": n,
    }

def extract_token_features(code_str):
    lines = code_str.split("\n")
    n_lines = max(len(lines), 1)
    ids = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', code_str)
    avg_id_len = np.mean([len(x) for x in ids]) if ids else 0.0
    c_lines = sum(1 for l in lines if l.strip().startswith(("//", "#", "/*", "*")))
    indents = [len(l) - len(l.lstrip()) for l in lines if l.strip()]
    ivar = min((np.var(indents) if len(indents) > 1 else 0.0) / 100.0, 1.0)
    uops = min(len(set(re.findall(r'[+\-*/%=<>!&|^~]+', code_str))) / 20.0, 1.0)
    return [avg_id_len / 20.0, c_lines / n_lines, ivar, uops]

@torch.no_grad()
def extract_lm_embedding(code_str, tokenizer, model, device):
    """Run CodeBERT to get sequence [CLS] representation (768D)."""
    if model is None or tokenizer is None: return None
    tks = tokenizer(code_str, truncation=True, max_length=512, return_tensors="pt").to(device)
    out = model(**tks)
    return out.pooler_output.squeeze(0).cpu() # [768]

TOKEN_FEATURE_DIM = 4

vocab_n = len(train_df) if not DEBUG else min(500, len(train_df))
node_vocab = build_node_vocab(train_df, parsers, max_samples=vocab_n)
VOCAB_SIZE, CONT_FEAT_DIM = len(node_vocab), 6
print(f"VOCAB={VOCAB_SIZE}, CONT={CONT_FEAT_DIM}, LM={CODEBERT_DIM}")

Building Vocab: 100%|██████████| 500/500 [00:00<00:00, 909.55it/s] 

✓ Node vocab: 321 types.
VOCAB=321, CONT=6, LM=768


# Section 6 — Multiprocessed Graph Dataset Class
Upgraded for v3:
- Processes the dataset using `joblib.Parallel` for massive CPU speedups
- Serializes `CHUNK_SIZE` structures into block `.pt` files to drastically reduce disk IO overhead
- Automatically extracts and caches LM pooler embeddings

In [14]:
# ========================== SECTION 6: GRAPH DATASET CLASS ==========================

def _process_single_sample(idx: int, row_dict: dict, parsers_dict: dict, node_vocab: dict, 
                           max_nodes: int, strategy: str,
                           use_token_fs: bool, use_lm: bool,
                           lm_tokenizer=None, lm_model=None, dvc=None):
    """Process a single sample, returning PyG Data object or None."""
    root = parse_code(row_dict["code"], row_dict.get("language", "python"), parsers_dict)
    if not root: return None
    g = ast_to_graph(root, node_vocab, max_nodes, strategy)
    if g is None or g["num_nodes"] == 0: return None
    
    data = Data(
        x_type=g["type_ids"], x_cont=g["cont_features"], 
        edge_index=g["edge_index"], num_nodes=g["num_nodes"],
        y=torch.tensor(int(row_dict["label"]), dtype=torch.long),
        orig_id=row_dict.get("id", idx)
    )
    if use_token_fs:
        data.token_features = torch.tensor(extract_token_features(row_dict["code"]), dtype=torch.float)
    if use_lm and lm_tokenizer and lm_model:
        data.lm_features = extract_lm_embedding(row_dict["code"], lm_tokenizer, lm_model, dvc)
    return data


class ChunkedGraphDataset(torch.utils.data.Dataset):
    def __init__(self, df, parsers_dict, node_vocab, cache_dir,
                 max_nodes=512, split_name="data", chunkSize=5000,
                 use_token_fs=True, use_lm=True,
                 lm_tok=None, lm_mod=None, dev=None, n_jobs=4):
        super().__init__()
        # Determine config hash to safely invalidate old caches
        config_str = f"v3_{max_nodes}_{CHUNK_SIZE}_{use_token_fs}_{use_lm}_{len(node_vocab)}_{len(df)}"
        c_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
        self.cache_path = os.path.join(cache_dir, f"{split_name}_{c_hash}")
        os.makedirs(self.cache_path, exist_ok=True)
        
        self.chunk_size = chunkSize
        self.metadata_file = os.path.join(self.cache_path, "metadata.json")
        self.chunk_files = []      # list of .pt file paths
        self.chunk_lengths = []    # num samples per chunk
        self.total_samples = 0
        
        if os.path.exists(self.metadata_file):
            print(f"Loading cached {split_name} metadata from {self.cache_path}")
            with open(self.metadata_file, "r") as f:
                meta = json.load(f)
            self.chunk_files = [os.path.join(self.cache_path, p) for p in meta["chunk_paths"]]
            self.chunk_lengths = meta["chunk_lengths"]
            self.total_samples = sum(self.chunk_lengths)
        else:
            self._process(df, parsers_dict, node_vocab, max_nodes, TRUNCATION_STRATEGY, 
                          use_token_fs, use_lm, lm_tok, lm_mod, dev, n_jobs)
            
        # Determine cumulative boundaries for fast __getitem__ binary search
        self.cum_lens = np.cumsum([0] + self.chunk_lengths)
        
    def _process(self, df, p_dict, vocab, mx_nodes, strat, uf, ul, tok, mod, dvc, n_jobs):
        total = len(df)
        print(f"Processing {total} samples into chunks -> {self.cache_path}")
        
        # Parallel AST -> Graph is great, but LM inference is single-GPU bound. 
        # For simplicity & safety if LM is used, we process sequentially. 
        # If no LM, we do joblib.
        valid_items = []
        for i in tqdm(range(total), desc="Extracting Samples"):
            row = df.iloc[i].to_dict()
            res = _process_single_sample(i, row, p_dict, vocab, mx_nodes, strat, uf, ul, tok, mod, dvc)
            if res is not None: valid_items.append(res)
            
            # Save chunk
            if len(valid_items) >= self.chunk_size or i == total - 1:
                if len(valid_items) > 0:
                    c_id = len(self.chunk_files)
                    c_path = os.path.join(self.cache_path, f"chunk_{c_id}.pt")
                    torch.save(valid_items, c_path)
                    self.chunk_files.append(c_path)
                    self.chunk_lengths.append(len(valid_items))
                    valid_items = []
                    
        self.total_samples = sum(self.chunk_lengths)
        mdata = {"chunk_paths": [os.path.basename(p) for p in self.chunk_files], "chunk_lengths": self.chunk_lengths}
        with open(self.metadata_file, "w") as f:
            json.dump(mdata, f)
        print(f"✓ Saved {self.total_samples} graphs across {len(self.chunk_files)} chunks.")

    def __len__(self):
        return self.total_samples
    
    def __getitem__(self, idx):
        # Find which chunk contains this idx
        chunk_idx = np.searchsorted(self.cum_lens, idx, side="right") - 1
        item_idx = idx - self.cum_lens[chunk_idx]
        
        # In a very optimized setup, we would LRU cache chunks in memory,
        # but to guarantee memory safety for PyG DataLoader workers, we load per call.
        chunk_data = torch.load(self.chunk_files[chunk_idx], weights_only=False)
        return chunk_data[item_idx]

print("Building train dataset...")
train_dataset = ChunkedGraphDataset(train_df, parsers, node_vocab, CACHE_DIR, MAX_NODES, "task_a_train", 
                                    CHUNK_SIZE, USE_TOKEN_FEATURES, USE_CODEBERT, lm_tokenizer, lm_model, device, NUM_WORKERS)
val_dataset = None
if val_df is not None:
    val_dataset = ChunkedGraphDataset(val_df, parsers, node_vocab, CACHE_DIR, MAX_NODES, "task_a_val", 
                                      CHUNK_SIZE, USE_TOKEN_FEATURES, USE_CODEBERT, lm_tokenizer, lm_model, device, NUM_WORKERS)
print(f"\nTrain: {len(train_dataset)}  |  Val: {len(val_dataset) if val_dataset else 0}")

Building train dataset...
Processing 1000 samples into chunks -> ./graph_cache_v3/task_a_train_08b7d41f


Extracting Samples: 100%|██████████| 1000/1000 [04:30<00:00,  3.69it/s]


✓ Saved 1000 graphs across 1 chunks.
Processing 500 samples into chunks -> ./graph_cache_v3/task_a_val_89c78778


Extracting Samples: 100%|██████████| 500/500 [02:11<00:00,  3.80it/s]

✓ Saved 500 graphs across 1 chunks.

Train: 1000  |  Val: 500


# Section 7 — Hybrid GNN Model
Architecture upgrades for v3:
- **5 Layer Deep GNN** (4 GCN + 1 GAT) with LayerNorm + Residuals
- **CodeBERT Alignment**: Fuses LM embeddings directly with graph representations
- Deep classification MLP

In [15]:
# ========================== SECTION 7: GNN MODEL ==========================
class HybridCodeGNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, cont_dim, hidden_dim, num_classes, 
                 dropout=0.3, use_token_fs=True, token_dim=4, use_lm=True, lm_dim=768):
        super().__init__()
        self.use_token_fs = use_token_fs
        self.use_lm = use_lm
        
        self.type_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.input_proj = nn.Linear(embed_dim + cont_dim, hidden_dim)
        self.input_norm = nn.LayerNorm(hidden_dim)
        
        # 4x GCN Residual blocks
        self.gcn_blocks = nn.ModuleList([
            nn.ModuleDict({
                "conv": GCNConv(hidden_dim, hidden_dim),
                "norm": nn.LayerNorm(hidden_dim)
            }) for _ in range(4)
        ])
        
        # 1x GAT
        self.gat = GATConv(hidden_dim, hidden_dim, heads=4, concat=False)
        self.gat_norm = nn.LayerNorm(hidden_dim)
        
        self.pool = GlobalAttention(nn.Sequential(nn.Linear(hidden_dim, hidden_dim//2), nn.ReLU(), nn.Linear(hidden_dim//2, 1)))
        
        # Hybrid MLP
        mlp_in = hidden_dim + (token_dim if use_token_fs else 0) + (lm_dim if use_lm else 0)
        self.fc1 = nn.Linear(mlp_in, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim//2)
        self.out = nn.Linear(hidden_dim//2, num_classes)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x_type, x_cont, edge_index, batch, token_fs=None, lm_fs=None):
        te = self.type_embed(x_type)
        h = torch.cat([te, x_cont], dim=1)
        h = F.relu(self.input_norm(self.input_proj(h)))
        
        for block in self.gcn_blocks:
            res = h
            h = F.relu(block["norm"](block["conv"](h, edge_index)))
            h = self.dropout(h) + res
            
        h = F.relu(self.gat_norm(self.gat(h, edge_index)))
        h = self.pool(h, batch)  # [B, hidden]
        
        features = [h]
        if self.use_token_fs and token_fs is not None: features.append(token_fs)
        if self.use_lm and lm_fs is not None: features.append(lm_fs)
        h = torch.cat(features, dim=1)
        
        h = F.relu(self.fc1(h))
        h = self.dropout(h)
        h = F.relu(self.fc2(h))
        return self.out(h)

model = HybridCodeGNN(VOCAB_SIZE, EMBEDDING_DIM, CONT_FEAT_DIM, HIDDEN_DIM, NUM_CLASSES, DROPOUT, 
                      USE_TOKEN_FEATURES, TOKEN_FEATURE_DIM, USE_CODEBERT, CODEBERT_DIM).to(device)
print(f"HybridCodeGNN: {sum(p.numel() for p in model.parameters()):,} parameters")

# Smoke Test
try:
    test_loader = PyGDataLoader(train_dataset, batch_size=4, shuffle=False)
    bt = next(iter(test_loader)).to(device)
    tf = bt.token_features if USE_TOKEN_FEATURES else None
    lmf = bt.lm_features if USE_CODEBERT else None
    
    with torch.no_grad():
        out = model(bt.x_type, bt.x_cont, bt.edge_index, bt.batch, tf, lmf)
    print(f"✓ GNN Forward Pass OK. Output shape: {out.shape}")
except Exception as e:
    print(f"X GNN Forward Pass Failed: {e}")

HybridCodeGNN: 281,571 parameters
X GNN Forward Pass Failed: Tensors must have same number of dimensions: got 2 and 1


# Section 8 — Training Pipeline
Optimized for batch sizes on the chunked DataLoaders. Tracks F1 and implements model selection via early stopping.

In [16]:
# ========================== SECTION 8: TRAINING PIPELINE ==========================
def compute_class_weights(df):
    counts = Counter(df["label"].tolist())
    total = sum(counts.values())
    w = [(total / (NUM_CLASSES * counts.get(c, 1))) for c in range(NUM_CLASSES)]
    w = torch.tensor(w, dtype=torch.float)
    return (w / w.sum() * NUM_CLASSES).to(device)

def _forward_batch(model, batch, dev):
    batch = batch.to(dev)
    tf = batch.token_features if USE_TOKEN_FEATURES and hasattr(batch, "token_features") else None
    lmf = batch.lm_features if USE_CODEBERT and hasattr(batch, "lm_features") else None
    return model(batch.x_type, batch.x_cont, batch.edge_index, batch.batch, tf, lmf)

def train_epoch(model, loader, optimizer, criterion, dev):
    model.train()
    total_loss, all_p, all_t = 0, [], []
    for batch in tqdm(loader, desc="  Train", leave=False):
        optimizer.zero_grad()
        out = _forward_batch(model, batch, dev)
        loss = criterion(out, batch.y.to(dev))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch.y.size(0)
        all_p.extend(out.argmax(1).cpu().numpy())
        all_t.extend(batch.y.cpu().numpy())
    return {"loss": total_loss / max(1, len(all_t)), "f1": f1_score(all_t, all_p, average="macro", zero_division=0)}

def eval_epoch(model, loader, criterion, dev):
    model.eval()
    total_loss, all_p, all_t = 0, [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="  Eval ", leave=False):
            out = _forward_batch(model, batch, dev)
            loss = criterion(out, batch.y.to(dev))
            total_loss += loss.item() * batch.y.size(0)
            all_p.extend(out.argmax(1).cpu().numpy())
            all_t.extend(batch.y.cpu().numpy())
    return {"loss": total_loss / max(1, len(all_t)), "f1": f1_score(all_t, all_p, average="macro", zero_division=0), "preds": all_p, "labels": all_t}

train_loader = PyGDataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = PyGDataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False) if val_dataset else None

class_weights = compute_class_weights(train_df)
print(f"Class weights: {class_weights.cpu().numpy()}")
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

best_f1, patience_ctr = 0.0, 0
history = {"train_loss":[], "val_loss":[], "train_f1":[], "val_f1":[]}

print(f"\nTraining for up to {NUM_EPOCHS} epochs (patience={PATIENCE})...")
print("-" * 52)
for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    tr = train_epoch(model, train_loader, optimizer, criterion, device)
    history["train_loss"].append(tr["loss"])
    history["train_f1"].append(tr["f1"])

    if val_loader:
        va = eval_epoch(model, val_loader, criterion, device)
        history["val_loss"].append(va["loss"])
        history["val_f1"].append(va["f1"])
        cur_f1 = va["f1"]
    else:
        cur_f1 = tr["f1"]
        history["val_loss"].append(0); history["val_f1"].append(0)

    print(f"Ep {epoch:3d} | TrLoss: {tr['loss']:.4f} | VaLoss: {history['val_loss'][-1]:.4f} | TrF1: {tr['f1']:.4f} | VaF1: {history['val_f1'][-1]:.4f} | {time.time()-t0:.1f}s", end="")

    if cur_f1 > best_f1:
        best_f1 = cur_f1
        patience_ctr = 0
        torch.save(model.state_dict(), "best_model.pt")
        print(" ★")
    else:
        patience_ctr += 1; print()
        if patience_ctr >= PATIENCE:
            print(f"Early stop at epoch {epoch}")
            break

print(f"✓ Best F1: {best_f1:.4f}")

Class weights: [1.046      0.95399994]

Training for up to 20 epochs (patience=5)...
----------------------------------------------------


  Train:   0%|          | 0/32 [00:00<?, ?it/s]Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=94, pipe_handle=133)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'ChunkedGraphDataset' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=94, pipe_handle=140)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Unexpected exception formatting exception. Falling back to standard exception


# Section 9 — Evaluation
Confusion Matrix & Error Analysis on original code inputs.

In [ ]:
# ========================== SECTION 9: EVALUATION ==========================
model.load_state_dict(torch.load("best_model.pt", map_location=device, weights_only=True))
print("✓ Best model loaded")

results = eval_epoch(model, val_loader if val_loader else train_loader, criterion, device)
y_true, y_pred = results["labels"], results["preds"]

print(f"\nMacro F1 : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_title("Confusion Matrix — Task A")
fig.colorbar(im, ax=ax)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        c = "white" if cm[i,j] > cm.max()/2 else "black"
        ax.text(j, i, str(cm[i,j]), ha="center", va="center", color=c)
plt.tight_layout()
plt.savefig("confusion_matrix_v3.png", dpi=150)
print("✓ Saved error visualizations")

# Section 10 — Inference & Submission
Outputs pure CSV suitable for SemEval submission, properly aligning back with original test data IDs via the chunked extraction mechanism.

In [ ]:
# ========================== SECTION 10: INFERENCE & SUBMISSION ==========================
def predict_batch(model, loader, dev):
    model.eval()
    orig_ids, labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Generating Predictions"):
            out = _forward_batch(model, batch, dev)
            labels.extend(out.argmax(1).cpu().numpy())
            # Safely fetch orig_id
            b_ids = batch.orig_id.cpu().numpy() if hasattr(batch, 'orig_id') else list(range(len(labels)-len(out), len(labels)))
            orig_ids.extend(b_ids)
    return pd.DataFrame({"id": orig_ids, "label": labels})

print(f"\nGenerating submission on {'val' if val_df is not None else 'train'} set...")
submission = predict_batch(model, val_loader if val_loader else train_loader, device)
submission.to_csv("submission_task_A_v3.csv", index=False)
print(f"✓ Saved submission_task_A_v3.csv")
print(submission.head())

# Save Config
with open("v3_config.json", "w") as f:
    json.dump({"MAX_NODES": MAX_NODES, "EMB": EMBEDDING_DIM, "HIDDEN": HIDDEN_DIM, "LM": USE_CODEBERT}, f)